# IEX vs Okta Checker
So sánh trạng thái agent trong Okta snapshot với lịch IEX (dữ liệu đã qua cleaner).

In [1]:
import pandas as pd
from datetime import datetime

# Hiển thị đủ số dòng và số cột mong muốn
pd.set_option("display.max_rows", 100)   # tối đa số dòng hiển thị
pd.set_option("display.max_columns", None)  # hiện tất cả các cột
pd.set_option("display.width", None)   # không giới hạn độ rộng

In [2]:
# Đọc dữ liệu từ file đã clean (iex_cleaner xuất ra)
iex_df = pd.read_excel('iex-data-extracted.xlsx')
okta_df = pd.read_csv('okta.csv')

# Xóa cột "Available On" nếu tồn tại
if "Available On" in okta_df.columns:
    okta_df = okta_df.drop(columns=["Available On"])

iex_df.head()

,IEX Id,Name,Shift,Activity,Start time,End time
0,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,Open Time,1:00 PM,2:35 PM
1,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,Break,2:35 PM,2:50 PM
2,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,Open Time,2:50 PM,5:30 PM
3,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,Lunch,5:30 PM,6:30 PM
4,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,Open Time,6:30 PM,8:35 PM


In [3]:
# Đổi tên cột Okta cho đồng bộ (nếu cần chỉnh lại tuỳ dataset)
okta_df = okta_df.rename(columns={
    'userName': 'Name',
    'status': 'Activity',
    'duration': 'Duration'
})
okta_df['CheckTime'] = datetime.now()
okta_df.head()

,Agent Name,Duration,State,Assigned Workitem Count,Agent Email,Queue Group / Routing Profile,Forecast Group,Manager Email,Business Location,CheckTime
0,"Bui, Hoang Cam Nhung",00:06:06,AVAILABLECHAT,1.0,hoangcamnhung.bui@concentrix.com,Chat_OD_EN_Car_Activity,GEN_GEN_EN_GCS_GLG_CHT,vuphuonguyen.nguyen1@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-21 05:44:06.744275
1,"Bui, Ngoc Thuan Vy",00:00:54,AVAILABLECHAT,2.0,ngocthuanvy.bui@concentrix.com,Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,thienkim.chau@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-21 05:44:06.744275
2,"Dang, Anh Trung",00:06:33,AVAILABLECHAT,1.0,anhtrung.dang@concentrix.com,Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,chihuy.vong@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-21 05:44:06.744275
3,"Dang, Phuong Tien",00:07:10,AVAILABLECHAT,1.0,phuongtien.dang1@concentrix.com,Chat_OD_EN_Car_Activity,GEN_GEN_EN_GCS_GLG_CHT,hoangkhoi.nguyen@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-21 05:44:06.744275
4,"Dinh, Thi Ngoc Han",00:02:19,AVAILABLECHAT,1.0,thingochan.dinh@concentrix.com,Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,dinhnhuhao.nguyen@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-21 05:44:06.744275


In [4]:
# Chuyển tất cả giá trị "Open Time" trong cột Activity thành "AVAILABLECHAT"
iex_df["Activity"] = iex_df["Activity"].replace("Open Time", "AVAILABLECHAT")

# Chuyển Start/End về datetime
iex_df['Start time'] = pd.to_datetime(iex_df['Start time'], errors='coerce')
iex_df['End time']   = pd.to_datetime(iex_df['End time'], errors='coerce')
iex_df.head()

C:\Users\huuchinh.nguyen\AppData\Local\Temp\ipykernel_23296\2382146022.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  iex_df['Start time'] = pd.to_datetime(iex_df['Start time'], errors='coerce')
C:\Users\huuchinh.nguyen\AppData\Local\Temp\ipykernel_23296\2382146022.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  iex_df['End time']   = pd.to_datetime(iex_df['End time'], errors='coerce')


,IEX Id,Name,Shift,Activity,Start time,End time
0,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,AVAILABLECHAT,2025-08-21 13:00:00,2025-08-21 14:35:00
1,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,Break,2025-08-21 14:35:00,2025-08-21 14:50:00
2,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,AVAILABLECHAT,2025-08-21 14:50:00,2025-08-21 17:30:00
3,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,Lunch,2025-08-21 17:30:00,2025-08-21 18:30:00
4,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,AVAILABLECHAT,2025-08-21 18:30:00,2025-08-21 20:35:00


In [5]:
import re

def clean_name(name: str) -> str:
    if pd.isna(name):
        return name
    # Thay dấu phẩy bằng khoảng trắng
    name = name.replace(",", " ")
    # Thêm khoảng trắng trước chữ in hoa (trừ chữ cái đầu)
    name = re.sub(r'(?<!^)(?=[A-Z])', ' ', name)
    # Chuẩn hoá khoảng trắng thừa
    name = " ".join(name.split())
    return name.strip()

# Chuẩn hoá cho cả IEX và Okta
iex_df["Name"] = iex_df["Name"].astype(str).map(clean_name)
okta_df["Agent Name"] = okta_df["Agent Name"].astype(str).map(clean_name)
#iex_df.head()
okta_df.head()

,Agent Name,Duration,State,Assigned Workitem Count,Agent Email,Queue Group / Routing Profile,Forecast Group,Manager Email,Business Location,CheckTime
0,Bui Hoang Cam Nhung,00:06:06,AVAILABLECHAT,1.0,hoangcamnhung.bui@concentrix.com,Chat_OD_EN_Car_Activity,GEN_GEN_EN_GCS_GLG_CHT,vuphuonguyen.nguyen1@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-21 05:44:06.744275
1,Bui Ngoc Thuan Vy,00:00:54,AVAILABLECHAT,2.0,ngocthuanvy.bui@concentrix.com,Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,thienkim.chau@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-21 05:44:06.744275
2,Dang Anh Trung,00:06:33,AVAILABLECHAT,1.0,anhtrung.dang@concentrix.com,Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,chihuy.vong@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-21 05:44:06.744275
3,Dang Phuong Tien,00:07:10,AVAILABLECHAT,1.0,phuongtien.dang1@concentrix.com,Chat_OD_EN_Car_Activity,GEN_GEN_EN_GCS_GLG_CHT,hoangkhoi.nguyen@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-21 05:44:06.744275
4,Dinh Thi Ngoc Han,00:02:19,AVAILABLECHAT,1.0,thingochan.dinh@concentrix.com,Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,dinhnhuhao.nguyen@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-21 05:44:06.744275


In [6]:
# Hàm chuẩn hoá tên để so sánh
def normalize_name(name: str) -> str:
    return ''.join(str(name).split()).upper()

# 1. So sánh giữa Okta và IEX
results = []
now = datetime.now()

for _, row in okta_df.iterrows():
    name_okta = row['Agent Name']
    activity_okta = row['State']
    workitem_count = row.get('Assigned Workitem Count', None)  # lấy cột này, tránh lỗi nếu cột không tồn tại
    
    # Skip các trường hợp không phải Available chat nhưng productive
    if str(activity_okta).strip().upper() != "AVAILABLECHAT" and pd.notna(workitem_count):
        continue  # bỏ qua agent này, sang vòng lặp tiếp theo
    
    # Tìm agent trong IEX (dùng normalize_name để so sánh)
    iex_agent = iex_df[iex_df['Name'].apply(normalize_name) == normalize_name(name_okta)]
    iex_now = iex_agent[(iex_agent['Start time'] <= now) & (iex_agent['End time'] >= now)]
    
    if not iex_now.empty:
        activity_iex = iex_now.iloc[0]['Activity']
        start_iex = iex_now.iloc[0]['Start time']
        end_iex = iex_now.iloc[0]['End time']
    else:
        activity_iex = 'N/A'
        start_iex = None
        end_iex = None
    
    results.append({
        'Agent': name_okta,   # giữ nguyên format tên từ Okta
        'Activity_Okta': activity_okta,
        'Activity_IEX': activity_iex,
        'Start_IEX': start_iex,
        'End_IEX': end_iex,
        'Match': str(activity_okta).strip().lower() == str(activity_iex).strip().lower()
    })


# 2. Bổ sung agent đang CÓ MẶT trong IEX (any activity hợp lệ) nhưng KHÔNG có trong Okta
iex_current = iex_df[
    (iex_df['Start time'] <= now) & 
    (iex_df['End time'] >= now)
].copy()

# Chỉ lấy những dòng có activity hợp lệ
iex_current = iex_current[iex_current['Activity'].notna() & (iex_current['Activity'].astype(str).str.strip() != '')]

# Nếu 1 agent có nhiều dòng tại thời điểm now, giữ dòng Start sớm nhất
iex_current = iex_current.sort_values(['Name', 'Start time']).drop_duplicates(subset=['Name'], keep='first')

# Danh sách agent đã có trong Okta (normalize để so sánh)
okta_agents_norm = set(okta_df['Agent Name'].apply(normalize_name))

for _, row in iex_current.iterrows():
    if normalize_name(row['Name']) not in okta_agents_norm:
        results.append({
            'Agent': row['Name'],  # giữ nguyên format tên từ IEX
            'Activity_Okta': 'N/A',
            'Activity_IEX': row['Activity'],
            'Start_IEX': row['Start time'],
            'End_IEX': row['End time'],
            'Match': False
        })

# 3. Kết quả cuối
result_df = pd.DataFrame(results)

# Danh sách activity cần loại bỏ
exclude_activities = ["Termination", "No Call/No Show", "Unpaid Leave", "PTO"]

# Lọc bỏ những dòng có Activity_IEX nằm trong danh sách exclude
result_df = result_df[~result_df['Activity_IEX'].isin(exclude_activities)]

result_df

,Agent,Activity_Okta,Activity_IEX,Start_IEX,End_IEX,Match
0,Bui Hoang Cam Nhung,AVAILABLECHAT,AVAILABLECHAT,2025-08-21 05:00:00,2025-08-21 06:15:00,True
1,Bui Ngoc Thuan Vy,AVAILABLECHAT,AVAILABLECHAT,2025-08-21 05:15:00,2025-08-21 07:00:00,True
2,Dang Anh Trung,AVAILABLECHAT,AVAILABLECHAT,2025-08-21 04:30:00,2025-08-21 07:00:00,True
3,Dang Phuong Tien,AVAILABLECHAT,AVAILABLECHAT,2025-08-21 05:00:00,2025-08-21 07:00:00,True
4,Dinh Thi Ngoc Han,AVAILABLECHAT,AVAILABLECHAT,2025-08-21 03:50:00,2025-08-21 06:00:00,True
5,Duong Thi Thuy Duong,AVAILABLECHAT,AVAILABLECHAT,2025-08-21 04:10:00,2025-08-21 06:00:00,True
6,Hoang Dai Hai,BREAK,AVAILABLECHAT,2025-08-21 04:30:00,2025-08-21 06:00:00,False
7,Huynh Ngoc Hue Trang,AVAILABLECHAT,AVAILABLECHAT,2025-08-21 02:50:00,2025-08-21 06:00:00,True
8,Huynh Quang Thuan,AVAILABLECHAT,N/A,NaT,NaT,False
9,Le Thanh Tung,AVAILABLECHAT,AVAILABLECHAT,2025-08-21 02:30:00,2025-08-21 06:15:00,True


In [7]:
# Xuất mismatch ra file Excel
mismatch_df = result_df[(result_df['Match'] == False) & (result_df["Activity_IEX"] != "N/A")]
mismatch_df.to_excel('iex_okta_mismatch.xlsx', index=False)
mismatch_df

,Agent,Activity_Okta,Activity_IEX,Start_IEX,End_IEX,Match
6,Hoang Dai Hai,BREAK,AVAILABLECHAT,2025-08-21 04:30:00,2025-08-21 06:00:00,False
14,Nguyen Dang Khoa,LOGIN,AVAILABLECHAT,2025-08-21 05:00:00,2025-08-21 07:25:00,False
20,Nguyen Thi Kim Ngan,ENDOFSHIFT,AVAILABLECHAT,2025-08-21 03:20:00,2025-08-21 06:00:00,False
28,Trieu Gia V Inh,TRAINING,AVAILABLECHAT,2025-08-21 02:15:00,2025-08-21 06:30:00,False
35,Dao Minh Nhat,N/A,AVAILABLECHAT,2025-08-21 05:00:00,2025-08-21 06:15:00,False
37,Ho Ngoc Huyen Trang,N/A,AVAILABLECHAT,2025-08-21 03:10:00,2025-08-21 06:00:00,False
38,Huy Nguyen Nhat Anh,N/A,AVAILABLECHAT,2025-08-21 05:00:00,2025-08-21 07:00:00,False
39,L E T H U Y T R A N G,N/A,AVAILABLECHAT,2025-08-21 05:00:00,2025-08-21 07:15:00,False
40,Lam Quoc Khang,N/A,AVAILABLECHAT,2025-08-21 04:15:00,2025-08-21 06:00:00,False
41,Le Cong Hoang,N/A,AVAILABLECHAT,2025-08-21 05:00:00,2025-08-21 07:00:00,False


In [8]:
# --- Export full comparison with both True/False ---
try:
    out_file = 'iex_okta_comparison.xlsx'
    result_df.to_excel(out_file, index=False)
    print(f'Saved full comparison to: {out_file}')
    result_df
except NameError as e:
    print('result_df is not defined. Please run the comparison cells above first.')
    raise


Saved full comparison to: iex_okta_comparison.xlsx
